Cell 1: Install Latest Libraries & Load Model (Step 1)
What this does (Plain English):
Before we can read the model's "mind," we need to load it into GPU memory. Here, we upgrade transformers to the latest 2026 v5 release to ensure full compatibility with the Qwen3.5 architecture. Notice that we use dtype=torch.bfloat16 instead of the old, deprecated torch_dtype argument. We also put the model in .eval() mode so it behaves deterministically without training randomness.

In [ ]:
# ==========================================================================
# Local environment check (replaces the old Colab `!pip install` cell)
# --------------------------------------------------------------------------
#   interpreter  : C:\Users\shuangboi\miniconda3\envs\torch_gpu\python.exe  (Python 3.12)
#   torch        : 2.14.1+cu130  -> ships the sm_120 kernels the RTX 5050 needs
#   transformers : 5.20.0        -> knows the `qwen3_5` architecture
#
# !! Never install torch with --index-url .../cu121 here: those wheels contain no
#    sm_120 kernels, and every CUDA op then fails with
#    "CUDA error: no kernel image is available for execution on the device".
# Set INSTALL_IF_MISSING = True only when rebuilding the environment from scratch.
# ==========================================================================
import importlib
import subprocess
import sys

INSTALL_IF_MISSING = False
TORCH_INDEX = "https://download.pytorch.org/whl/cu130"

# import-name -> pip-name
REQUIRED = {
    "torch": "torch",
    "torchvision": "torchvision",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "numpy": "numpy",
}


def _missing_packages():
    missing = []
    for module, package in REQUIRED.items():
        try:
            importlib.import_module(module)
        except ImportError:
            missing.append(package)
    return missing


missing = _missing_packages()

if missing and INSTALL_IF_MISSING:
    print(f"Installing missing packages: {missing}")
    if "torch" in missing:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "torch", "torchvision", "--index-url", TORCH_INDEX]
        )
    third_party = [p for p in missing if p != "torch"]
    if third_party:
        subprocess.check_call([sys.executable, "-m", "pip", "install", *third_party])
    missing = _missing_packages()

if missing:
    raise RuntimeError(
        f"Missing packages {missing}. Re-run this cell with INSTALL_IF_MISSING = True, or install them with:\n"
        f"  python -m pip install torch torchvision --index-url {TORCH_INDEX}\n"
        f"  python -m pip install {' '.join(p for p in missing if p != 'torch')}"
    )

import torch

print(f"Python     : {sys.executable}")
print(f"PyTorch    : {torch.__version__}  (compiled for CUDA {torch.version.cuda})")
print(f"CUDA ready : {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("PyTorch cannot see the GPU. Do not install cu121/cu124 wheels on this machine.")

# A build can report "CUDA available" yet still lack kernels for this GPU architecture
# (that is exactly what cu121 did here), so prove it with a real GPU op first.
_probe = torch.randn(64, 64, device="cuda", dtype=torch.bfloat16)
_probe = _probe @ _probe
torch.cuda.synchronize()
del _probe
torch.cuda.empty_cache()

major, minor = torch.cuda.get_device_capability(0)
print(f"GPU        : {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB)")
print(f"Compute cap: sm_{major}{minor}")
print(f"Archs      : {torch.cuda.get_arch_list()}")
print("CUDA bfloat16 matmul smoke test: OK")


### Local Environment Setup (this notebook is 100% local now — no Colab, no local-runtime bridge)

Everything runs inside the conda environment **`torch_gpu`** (`C:\Users\shuangboi\miniconda3\envs\torch_gpu\python.exe`, Python 3.12), which is also this notebook's kernel. The old Colab "connect to a local runtime" workflow is gone — just pick the kernel and Run All.

```powershell
conda activate torch_gpu
python -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130
python -m pip install transformers accelerate scikit-learn matplotlib
```

**Why `cu130` and not `cu121`?** The RTX 5050 is a Blackwell card (`sm_120`). The `cu121` wheels contain no `sm_120` kernels, so every CUDA op dies with:

```text
RuntimeError: CUDA error: no kernel image is available for execution on the device
```

That failure is exactly why the model used to land on the CPU (`device: 'cpu'`). The `cu130` build lists `sm_120` in `torch.cuda.get_arch_list()`, and `transformers>=5.0` is needed because Qwen3.5's architecture (`qwen3_5`) does not exist in older releases.


In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, Qwen3_5ForConditionalGeneration

print(f"Using PyTorch {torch.__version__} and NumPy {np.__version__}")

# Official Hugging Face repo ID for Qwen3.5 0.8B (24 layers, 1024 hidden size)
MODEL_ID = "Qwen/Qwen3.5-0.8B"

print("Loading tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Qwen3.5 ships as an image-text-to-text checkpoint (Qwen3_5ForConditionalGeneration),
# so AutoModelForCausalLM does not cover its architecture. We only feed text here, so the
# vision tower stays idle while the language-model hidden states remain fully accessible.
model = Qwen3_5ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    device_map="auto",
    dtype=torch.bfloat16,
)

# Switch to evaluation mode
model.eval()
print(f"Success! Model loaded on: {model.device}")
print(f"Total Transformer Layers: {model.config.text_config.num_hidden_layers}")
print(f"Hidden Vector Dimension: {model.config.text_config.hidden_size}")


Cell 2: Extract Mid-Layer Hidden States (Step 2)
What this does (Plain English):
This is the core magic. Normally, you ask an LLM a question and it generates words one by one. We are not going to let it generate a single word.
Instead, we feed the prompt into the model and run just one forward pass (the "Prefill" stage) with output_hidden_states=True. As the model finishes reading the prompt and prepares to speak, we reach into Layer 15 (roughly 62% deep into its 24-layer brain) and grab the vector of the very last token (-1). This single 1024-number vector captures the model's internal "confidence vs. hesitation" right before it opens its mouth.

In [ ]:
# We create two types of order emails based on your business scenario:
# Label 0 = Answer is clearly in the text (Safe / Grounded)
# Label 1 = Answer is MISSING, so the model is tempted to guess/hallucinate (Danger!)

grounded_samples = [
    ("Order #101: 2x Mech Keyboards. Total price is $150. Shipped to Helsinki.", "What is the total price?"),
    ("Hi, my invoice #992 shows an amount due of 450 EUR for the server rack.", "What is the amount due?"),
    ("Confirmation: You bought 5 licenses. Grand total: $1,200 paid via Visa.", "What is the grand total?"),
    ("Receipt for Order #404: 1x Monitor ($300) + Shipping ($20). Total = $320.", "What is the total cost?"),
    ("Client email: We agreed on a final project fee of 2,500 USD.", "What is the final project fee?"),
    ("Order summary: 10x USB cables. Total billed to card: 85 EUR.", "How much was billed?"),
    ("Quote #88: Total price including VAT is 640 EUR.", "What is the total price?"),
    ("Payment received: $99.00 for annual Pro subscription.", "How much was the payment?"),
    ("Invoice details: Net amount 500 EUR, Tax 100 EUR. Total payable: 600 EUR.", "What is the total payable?"),
    ("Refund request for Order #12: Original purchase price was $45.", "What was the purchase price?"),
]

ungrounded_samples = [
    ("Order #102: Please ship 2x Mech Keyboards to Helsinki ASAP! Thanks.", "What is the total price?"),
    ("Hi, I lost my invoice #993 for the server rack. Can you resend it?", "What is the amount due?"),
    ("Confirmation: You bought 5 licenses. Paid via corporate Visa card.", "What is the grand total?"),
    ("Order #405: 1x Monitor. Please let me know how much shipping will cost.", "What is the total cost?"),
    ("Client email: We are still negotiating the final project fee.", "What is the final project fee?"),
    ("Order summary: 10x USB cables. Billing is still pending approval.", "How much was billed?"),
    ("Quote #89: Please send me the price including VAT when ready.", "What is the total price?"),
    ("Subscription active: You are on the annual Pro plan.", "How much was the payment?"),
    ("Invoice details: Tax rate is 20%. The net amount is missing from the sheet.", "What is the total payable?"),
    ("Refund request for Order #13: The item arrived broken, I want my money back.", "What was the purchase price?"),
]

# Combine into a mini dataset (20 samples for quick demo; use 100+ in real life)
dataset = [(ctx, q, 0) for ctx, q in grounded_samples] + \
          [(ctx, q, 1) for ctx, q in ungrounded_samples]

# Qwen3.5-0.8B has 24 layers. Output tuple has 25 items (Index 0 is embeddings, 1..24 are layers).
# Layer 15 is right around the ~60% sweet spot where knowledge boundaries show up best.
TARGET_LAYER = 15

def extract_mid_layer_vector(context: str, question: str) -> np.ndarray:
    """
    Runs a single forward pass (no text generation!) and grabs the hidden state
    of the very last prompt token at TARGET_LAYER.
    """
    messages = [
        {"role": "system", "content": "Extract the exact requested value from the email."},
        {"role": "user", "content": f"Email: {context}\nQuestion: {question}"}
    ]

    # 2026 HF Official Chat Template API: tokenize and return a dictionary directly
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    # Run the model forward pass WITHOUT generating new tokens
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    # outputs.hidden_states is a tuple of tensors: (batch_size, sequence_length, hidden_size)
    layer_tensor = outputs.hidden_states[TARGET_LAYER]

    # Grab batch 0, the LAST token (-1), and all 1024 hidden dimensions (:)
    # We convert from bfloat16 to float32 so NumPy / Scikit-Learn can read it
    last_token_vector = layer_tensor[0, -1, :].to(torch.float32).cpu().numpy()
    return last_token_vector

# Extract vectors for all samples
X_vectors = []
y_labels = []

print(f"Extracting vectors from Layer {TARGET_LAYER}...")
for ctx, q, label in dataset:
    vec = extract_mid_layer_vector(ctx, q)
    X_vectors.append(vec)
    y_labels.append(label)

X = np.array(X_vectors) # Shape: (20, 1024)
y = np.array(y_labels)  # Shape: (20,)
print(f"Done! Extracted feature matrix of shape: {X.shape}")

Cell 3: Train the Linear Probe ("The Lie Detector") (Step 3)
What this does (Plain English):
Now we have a table of 20 rows and 1024 columns. Half of the rows come from prompts where the model has the answer; the other half come from prompts where the answer is missing. We feed this into a standard LogisticRegression from Scikit-Learn. It draws a flat mathematical boundary (a hyperplane) through that 1024-dimensional space to separate "I see the fact in the email" from "I don't see it, I'm about to guess.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Build a tiny pipeline: normalize the 1024 numbers, then run Logistic Regression
probe = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=0.1, max_iter=1000)
)

# Evaluate with 5-Fold Cross Validation to see if the signal is real
scores = cross_val_score(probe, X, y, cv=5, scoring="accuracy")
print(f"5-Fold Cross-Validation Accuracy: {scores.mean() * 100:.1f}% (Individual folds: {scores})")

# Fit the probe on all data so we can use it as our runtime circuit breaker
probe.fit(X, y)
print("Linear Probe trained and ready!")

Cell 4: The "Pull the Plug" Circuit Breaker in Action (Bonus Demo)
What this does (Plain English):
Here is how you actually force the model to output "blank" (None) in production without touching its weights or doing complex activation steering.

A new email arrives.

We run the fast prefill pass to check Layer 15's hidden state.

Our probe calculates the Hallucination Risk Score (takes less than 0.1 milliseconds).

If the risk is above 0.60, we physically block model.generate() and return {"answer": None, "status": "ABSTAINED_MISSING_INFO"}. The LLM never even gets the chance to hallucinate.

In [ ]:
def safe_extract_order_info(email_text: str, question: str, threshold: float = 0.60):
    """
    Checks the mid-layer probe BEFORE allowing the LLM to generate text.
    If the model is internally 'hesitating', we pull the plug and return None.
    """
    # Step A: Read the model's mind at Layer 15
    vec = extract_mid_layer_vector(email_text, question).reshape(1, -1)

    # Step B: Get probability of Class 1 (Missing Info / Hallucination Risk)
    risk_score = probe.predict_proba(vec)[0, 1]

    print(f"\n--- Testing Email: '{email_text}' ---")
    print(f"[Mid-Layer Probe] Hallucination Risk Score: {risk_score:.3f} (Threshold: {threshold})")

    # Step C: Hard Cutoff! Force a 'Blank' output if risk is too high
    if risk_score >= threshold:
        print("🛑 CIRCUIT BREAKER TRIGGERED: Info missing in context. Pulling the plug!")
        return {"status": "BLOCKED_BY_PROBE", "answer": None, "risk": round(float(risk_score), 3)}

    # Step D: If risk is low, let the model generate normally
    print("✅ PROBE PASSED: Context is grounded. Allowing LLM to generate...")
    messages = [
        {"role": "system", "content": "Extract the exact requested value from the email. Be brief."},
        {"role": "user", "content": f"Email: {email_text}\nQuestion: {question}"}
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        gen_tokens = model.generate(**inputs, max_new_tokens=20, do_sample=False)

    # Slice off the prompt tokens and decode only the newly generated answer
    input_length = inputs["input_ids"].shape[1]
    answer_text = tokenizer.decode(gen_tokens[0, input_length:], skip_special_tokens=True).strip()

    return {"status": "SUCCESS", "answer": answer_text, "risk": round(float(risk_score), 3)}

# Test Case 1: Clear price present in email
res1 = safe_extract_order_info(
    "Order #777: 3x Office Chairs. Total amount charged is 890 EUR.",
    "What is the total amount charged?"
)
print("Result 1:", res1)

# Test Case 2: No price in email (Normally, small models might guess a number or ramble)
res2 = safe_extract_order_info(
    "Order #778: Please send 3x Office Chairs to our Espoo warehouse tomorrow.",
    "What is the total amount charged?"
)
print("Result 2:", res2)

Cell 5: Break the Naive Probe with "Decoy Numbers" (Adversarial Test)
What this does (Plain English):
We are going to feed our existing probe three tricky emails. None of them contain the actual order price, so the probe should block all of them (Risk >= 0.60). However, we stuffed them with decoy numbers: street addresses (1500), weights (450 kg), zip codes (00100), and even a fake currency reference (500 EUR limit, not the order price). Let's see if our naive probe gets fooled and accidentally lets the LLM hallucinate!

In [ ]:
# Tricky emails with NO actual order price, but full of distracting numbers
adversarial_tests = [
    # Trap 1: Order ID (890), Quantity (15), Street Number (1500), Zip Code (00100)
    (
        "Order #890: Please deliver 15 ergonomic chairs to 1500 Mannerheimintie, 00100 Helsinki.",
        "What is the total amount charged?"
    ),
    # Trap 2: Part Number (4090), Weight (450 kg), Phone Extension (2500)
    (
        "Shipment for Part #4090 weighs 450 kg. Call warehouse ext 2500 upon arrival.",
        "What is the total price?"
    ),
    # Trap 3: Currency decoy! Mentions "500 EUR" as a policy rule, NOT the actual order cost
    (
        "Note: Any order over 500 EUR requires manager approval. Please send the invoice for our 3 laptops.",
        "What is the invoice total for the laptops?"
    )
]

print("=== 🚨 TESTING NAIVE PROBE AGAINST DECOY NUMBERS ===")
for email_text, question in adversarial_tests:
    res = safe_extract_order_info(email_text, question, threshold=0.60)
    print("Output:", res)

Cell 6: Upgrade Dataset with "Hard Negatives" & Retrain
What this does (Plain English):
To stop the probe from lazily looking for any number, we must add Hard Negatives to Class 1 (Missing Info). By showing the linear classifier examples where numbers and currency words exist—yet the requested price is still missing—we force the logistic regression weights to ignore surface-level number features and lock onto the exact neurons that represent "Semantic Match between Question and Context."

In [ ]:
# Hard Negatives: Packed with dates, IDs, addresses, weights, and decoy prices,
# but the ACTUAL answer to the question is still missing (Label = 1).
hard_negative_samples = [
    ("Order #850: Deliver 20 monitors to 1200 Aleksanterinkatu, 00100 Helsinki by 15:00.", "What is the total price?"),
    ("Invoice #4020 is still being drafted for the 500 kg server rack.", "What is the amount due?"),
    ("Company policy: Orders above $1,000 need CFO sign-off. Please quote us for 10 desks.", "What is the grand total?"),
    ("Item SKU-9900 (300 grams). Shipping distance is 450 km to Tampere.", "What is the total cost?"),
    ("We had a budget of 3,000 USD last year, but haven't priced this new project yet.", "What is the final project fee?"),
    ("Card ending in 4455 was declined 3 times on 2026-10-12. Please send a new link.", "How much was billed?"),
    ("Discount code SAVE50 gives 50 EUR off, please send the final quote with VAT.", "What is the total price?"),
    ("You have 365 days left on your 2026 Pro plan (Account ID: 889900).", "How much was the payment?"),
    ("Tax ID: FI-1234567. Net weight 800 kg, 24 pallets.", "What is the total payable?"),
    ("Refund policy: 14 days return window, $25 restocking fee applies. My item broke.", "What was the purchase price?"),
]

# Also add 10 more Grounded samples (Label = 0) that ALSO have decoy numbers,
# so the probe doesn't overreact and block emails just because they have addresses/IDs!
hard_positive_samples = [
    ("Order #850 to 1200 Aleksanterinkatu: 20 monitors. Total price is 4,200 EUR.", "What is the total price?"),
    ("Invoice #4020 for the 500 kg server rack: Amount due is $1,850.", "What is the amount due?"),
    ("Orders above $1,000 need sign-off. Your 10 desks come to a grand total of $2,400.", "What is the grand total?"),
    ("Item SKU-9900 (300g) shipped 450 km. Total cost billed: 75 EUR.", "What is the total cost?"),
    ("Out of your 3,000 USD budget, the agreed final project fee is 2,100 USD.", "What is the final project fee?"),
    ("Card ending in 4455 was billed 340 EUR on 2026-10-12.", "How much was billed?"),
    ("After applying SAVE50 (50 EUR off), your total price is 490 EUR.", "What is the total price?"),
    ("Account ID 889900 (365 days Pro plan): Payment received was $299.", "How much was the payment?"),
    ("Tax ID FI-1234567, 800 kg cargo: Total payable is 3,600 EUR.", "What is the total payable?"),
    ("Under our 14-day policy ($25 fee), your original purchase price was $180.", "What was the purchase price?"),
]

# 1. Expand dataset to 40 samples (20 Grounded vs 20 Ungrounded with Hard Negatives)
upgraded_dataset = dataset + \
                   [(ctx, q, 0) for ctx, q in hard_positive_samples] + \
                   [(ctx, q, 1) for ctx, q in hard_negative_samples]

print(f"Extracting vectors for {len(upgraded_dataset)} upgraded samples at Layer {TARGET_LAYER}...")
X_up = np.array([extract_mid_layer_vector(c, q) for c, q, _ in upgraded_dataset])
y_up = np.array([label for _, _, label in upgraded_dataset])

# 2. Retrain the probe on the hardened dataset
scores_up = cross_val_score(probe, X_up, y_up, cv=5, scoring="accuracy")
print(f"Upgraded 5-Fold CV Accuracy: {scores_up.mean() * 100:.1f}% (Folds: {scores_up})")

probe.fit(X_up, y_up)
print("\n=== 🛡️ RE-TESTING UPGRADED PROBE AGAINST THE EXACT SAME TRAPS ===")
for email_text, question in adversarial_tests:
    res = safe_extract_order_info(email_text, question, threshold=0.60)
    print("Output:", res)

Cell 7: Scan All 24 Layers & Plot the "Truthfulness Curve"
What this does (Plain English):
Instead of guessing that Layer 15 is the best, we extract the last-token vector from every single layer (Layer 0 to Layer 24) in one pass. Then we train 25 separate Logistic Regression probes and plot their 5-fold cross-validation accuracy. You will visibly see the "Anatomy of a Transformer":

Early Layers (0–5): Low accuracy. The model is just parsing grammar and raw tokens.

Middle-to-Late Layers (12–19): Peak accuracy ("The Golden Zone"). Induction heads have finished cross-referencing the question against the email context.

Final Layers (22–24): Slight drop or instability. The model shifts focus from "semantic truth" to "formatting the next word's syntax."

In [ ]:
import matplotlib.pyplot as plt

def extract_all_layers_at_once(context: str, question: str) -> np.ndarray:
    """
    Runs ONE forward pass and returns a matrix of shape (25, 1024):
    Index 0 = Embedding layer, Index 1..24 = Transformer layers 1..24.
    """
    messages = [
        {"role": "system", "content": "Extract the exact requested value from the email."},
        {"role": "user", "content": f"Email: {context}\nQuestion: {question}"}
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    # Stack all 25 hidden states at the last token (-1) -> Shape: (25, 1024)
    all_layers = torch.stack([h[0, -1, :] for h in outputs.hidden_states])
    return all_layers.to(torch.float32).cpu().numpy()

print("Scanning all 25 layers across all 40 upgraded samples...")
# Shape of X_all_layers: (40 samples, 25 layers, 1024 hidden_dim)
X_all_layers = np.array([extract_all_layers_at_once(c, q) for c, q, _ in upgraded_dataset])
y_all = np.array([label for _, _, label in upgraded_dataset])

layer_accuracies = []
best_layer = 0
best_acc = 0.0

for layer_idx in range(25):
    X_layer = X_all_layers[:, layer_idx, :]
    layer_probe = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=1000))
    scores = cross_val_score(layer_probe, X_layer, y_all, cv=5, scoring="accuracy")
    mean_acc = scores.mean() * 100
    layer_accuracies.append(mean_acc)

    if mean_acc >= best_acc and layer_idx > 0:
        best_acc = mean_acc
        best_layer = layer_idx

print(f"\n🏆 Golden Layer Found: Layer {best_layer} with {best_acc:.1f}% CV Accuracy!")

# Plot the curve
plt.figure(figsize=(10, 4.5))
plt.plot(range(25), layer_accuracies, marker='o', linewidth=2, label="Probe CV Accuracy (%)")
plt.axvline(x=best_layer, color='red', linestyle='--', label=f"Best Layer ({best_layer})")
plt.title("Qwen3.5-0.8B: Hallucination Detection Accuracy Across All 24 Layers")
plt.xlabel("Layer Index (0 = Embeddings, 1..24 = Transformer Blocks)")
plt.ylabel("5-Fold CV Accuracy (%)")
plt.xticks(range(0, 25))
plt.ylim(40, 105)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

Cell 8: The Ultimate Trap — "Historical Price vs. Current Price"
What this does (Plain English):
Now we test the hardest business edge-case: Entity/Temporal Confusion.
In these emails, there is a real order price (not a street number or policy limit!), and it is paid in real currency (EUR/USD). However, it belongs to last year's order or a cancelled quote, while the 2026 renewal price is still missing. Let's see if our probe trained only on general decoy numbers gets tricked by a real historical price, and how adding just 4 temporal contrast pairs fixes it.

In [ ]:
# Ultimate Traps: Real prices for the SAME product, but wrong timeframe/status!
ultimate_temporal_traps = [
    (
        "Hi, we paid 400 EUR for the annual license in 2025. Please let us know the new renewal price for 2026.",
        "What is the 2026 renewal price?"
    ),
    (
        "Quote #10 (Cancelled): Total was $1,200. We are still waiting for the revised Quote #11 total.",
        "What is the total for revised Quote #11?"
    ),
    (
        "Competitor X charged us 850 EUR for this server, can you beat their price? Send us your quote.",
        "What is our quoted price for the server?"
    )
]

print("=== 💀 TESTING CURRENT PROBE AGAINST ULTIMATE TEMPORAL TRAPS ===")
for email_text, question in ultimate_temporal_traps:
    res = safe_extract_order_info(email_text, question, threshold=0.60)
    print("Output:", res)

In [ ]:
# GPT's Counterfactual Test:
# Pair each Trap (Missing target price -> Should be >= 0.60)
# with its exact Counterfactual Twin (Target price ADDED -> Should drop < 0.60!)

counterfactual_pairs = [
    # Pair 1: Temporal Trap vs. Temporal Grounded
    (
        "TRAP (No 2026 price)",
        "Hi, we paid 400 EUR for the annual license in 2025. Please let us know the new renewal price for 2026.",
        "What is the 2026 renewal price?",
        "SHOULD BLOCK (>= 0.6)"
    ),
    (
        "COUNTERFACTUAL (Has 2026 price)",
        "Hi, we paid 400 EUR for the annual license in 2025. The new renewal price for 2026 is 450 EUR.",
        "What is the 2026 renewal price?",
        "SHOULD PASS (< 0.6)"
    ),

    # Pair 2: Cancelled Quote vs. Revised Quote Grounded
    (
        "TRAP (No Quote #11 total)",
        "Quote #10 (Cancelled): Total was $1,200. We are still waiting for the revised Quote #11 total.",
        "What is the total for revised Quote #11?",
        "SHOULD BLOCK (>= 0.6)"
    ),
    (
        "COUNTERFACTUAL (Has Quote #11 total)",
        "Quote #10 (Cancelled): Total was $1,200. The revised Quote #11 total is $950.",
        "What is the total for revised Quote #11?",
        "SHOULD PASS (< 0.6)"
    ),

    # Pair 3: Competitor Price vs. Our Quote Grounded
    (
        "TRAP (No our quote)",
        "Competitor X charged us 850 EUR for this server, can you beat their price? Send us your quote.",
        "What is our quoted price for the server?",
        "SHOULD BLOCK (>= 0.6)"
    ),
    (
        "COUNTERFACTUAL (Has our quote)",
        "Competitor X charged us 850 EUR for this server, so our quoted price for the server is 790 EUR.",
        "What is our quoted price for the server?",
        "SHOULD PASS (< 0.6)"
    ),
]

print("=== 🔬 RUNNING GPT'S COUNTERFACTUAL ACID TEST (Layer 15 Probe) ===")
for tag, email_text, question, expected in counterfactual_pairs:
    vec = extract_mid_layer_vector(email_text, question).reshape(1, -1)
    risk_score = probe.predict_proba(vec)[0, 1]
    decision = "BLOCKED 🛑" if risk_score >= 0.60 else "PASSED ✅"
    print(f"[{tag}]")
    print(f"  Expected: {expected} | Actual Risk: {risk_score:.3f} -> {decision}\n")

Below we call the `safe_extract_order_info` function defined earlier. This shows the probe's risk score (block vs. false-pass), and also checks: if the probe does let it through, can this 0.8B model — with thinking mode off and capped at 20 tokens — actually compute 3 × 200 = 600?

Cell 10: The "Implicit Math" Boundary Test (Literal Extraction vs. Solvability)
What this does (Plain English):
Up to now, every "Safe" (Class 0) email in our training set had the final number explicitly written down (e.g., Total = $320).
Now we test Implicit Grounding: what if the email contains all the raw ingredients to calculate the total (3 chairs × 200 EUR), or Monitor ($300) + Shipping ($20), but never writes down the final sum?
We also include a Partial-Math Trap (3 chairs + 50 EUR shipping, where the chair price is missing) to see if the probe can tell the difference between "complete math ingredients" and "incomplete math ingredients."


In [ ]:
# Testing the boundary between "Literal Span Presence" and "Mathematical Solvability"
implicit_math_tests = [
    # Case 1: Multiplication required (3 * 200 = 600 EUR). No explicit "Total" number written.
    (
        "IMPLICIT MULTIPLICATION (Complete Info)",
        "Order #900: Please ship 3x Office Chairs at 200 EUR each.",
        "What is the total amount charged?"
    ),
    # Case 2: Addition required (300 + 20 = $320). Compare this to Cell 2 where "Total = $320" was written!
    (
        "IMPLICIT ADDITION (Complete Info)",
        "Receipt for Order #901: 1x Monitor ($300) plus Shipping ($20).",
        "What is the total cost?"
    ),
    # Case 3: Partial Math Trap! Has shipping price (50 EUR) and quantity (3x), but NO unit price for chairs.
    (
        "INCOMPLETE MATH TRAP (Missing Unit Price)",
        "Order #902: 3x Office Chairs, plus 50 EUR for express shipping.",
        "What is the total amount charged?"
    )
]

print("=== 🧮 TESTING IMPLICIT MATH & SOLVABILITY BOUNDARY ===")
for tag, email_text, question in implicit_math_tests:
    print(f"\n[{tag}]")
    res = safe_extract_order_info(email_text, question, threshold=0.60)
    print("Output:", res)

### Check local GPU (CUDA) availability and configuration
This cell confirms whether your local PyTorch install correctly detects the NVIDIA GPU and CUDA driver (the environment previously reported `device: 'cpu'`).


In [ ]:
# ==========================================================================
# Consolidated local GPU check (replaces 4 duplicated Colab-era check cells)
# ==========================================================================
import sys

import torch

print("=== Local GPU status ===")
print(f"Python executable   : {sys.executable}")
print(f"Python version      : {sys.version.split()[0]}")
print(f"PyTorch version     : {torch.__version__} (compiled for CUDA {torch.version.cuda})")
print(f"CUDA (GPU) available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    print(f"GPU device          : {torch.cuda.get_device_name(0)}")
    print(f"Compute capability  : sm_{major}{minor}")
    print(f"Compiled arch list  : {torch.cuda.get_arch_list()}")
else:
    print("\n🛑 No CUDA device detected. Run this cell with the project environment (conda `torch_gpu`):")
    print("   and make sure torch was installed from https://download.pytorch.org/whl/cu130")


In [ ]:
# Note on `sm_120` / compatibility warnings:
# The RTX 5050 is a Blackwell card (sm_120). Wheels built for cu121/cu124 do NOT ship
# sm_120 kernels -- that is exactly why this notebook pins the cu130 PyTorch build.
# With cu130 there is no "not compatible" warning and the probe trains on the GPU.


In [ ]:
import time
from sklearn.metrics import confusion_matrix, accuracy_score

# 16 never-before-seen "dirty-format" test emails (newlines, quote markers >, phone numbers, VAT signature blocks).
# Label 0 = the answer is clearly present in the email (Mute should PASS).
# Label 1 = the target answer is missing; only decoy numbers remain (Mute should BLOCK before generation).
messy_holdout_emails = [
    # --- 8 dirty emails with a real answer (Label = 0: should pass) ---
    (
        "Hi team,\nWe finalized the catering order for Friday's workshop.\n> Quote ref: CAT-2026-88\n> Deposit required: 350 EUR (due by Thursday)\n--\nBest, Maria | Tel: +358 40 123 4567 | VAT: FI99887766",
        "What is the deposit required?", 0
    ),
    (
        "Forwarded message:\nFrom: Logistics <ship@nordicfreight.fi>\nSubject: Pallet #4412 Customs\nCustoms clearance fee came to $125.00. Storage was waived ($0).\nOffice: Porkkalankatu 20, 00180 Helsinki.",
        "What was the customs clearance fee?", 0
    ),
    (
        "Hello,\nRegarding your SaaS Enterprise renewal (Contract #771):\nLast year you paid $4,000. After the 10% loyalty discount, your 2026 annual subscription cost is $3,600.\nRegards, Billing Desk (Ext. 808)",
        "What is the 2026 annual subscription cost?", 0
    ),
    (
        "Ticket #5092 — Damaged Dell UltraSharp\nWe approved your partial refund of 180 EUR. It will hit card *8821 within 5 business days.\nSupport HQ | 2400 Espoo",
        "What is the approved refund amount?", 0
    ),
    (
        "Hi Alex,\nFor the 12x ThinkPad laptops, express air freight is 220 EUR. Standard ground takes 7 days.\nLet me know by 17:00.\n-- Janne, Account Mgr (#ID-302)",
        "How much is express air freight?", 0
    ),
    (
        "Invoice #INV-9001 Summary:\nConsulting hours: 40h. Total balance due: 4,800 EUR.\nPlease wire to IBAN FI2112345600000785 before Nov 30.",
        "What is the total balance due?", 0
    ),
    (
        "Re: Booth rental for Slush 2026\nThe early-bird booth fee is 1,950 EUR (includes 4 passes). Penalty for late cancellation after Oct 15 is 500 EUR.",
        "What is the early-bird booth fee?", 0
    ),
    (
        "Automated Receipt:\nAWS Cloud usage for Sept 2026 (Account 449102938). Total charged to Visa *1109: $842.50.",
        "What was the total charged to the card?", 0
    ),

    # --- 8 trap emails missing the target answer (Label = 1: should block) ---
    (
        "Hi team,\nWe finalized the headcount (45 people) for Friday's workshop.\n> Quote ref: CAT-2026-88\n> Caterer is still calculating the deposit required.\n--\nBest, Maria | Tel: +358 40 123 4567 | VAT: FI99887766",
        "What is the deposit required?", 1
    ),
    (
        "Forwarded message:\nFrom: Logistics <ship@nordicfreight.fi>\nSubject: Pallet #4412 (Weight: 620 kg, Value: $8,500)\nCustoms is still holding the shipment; they haven't told us the clearance fee yet.\nOffice: Porkkalankatu 20, 00180 Helsinki.",
        "What was the customs clearance fee?", 1
    ),
    (
        "Hello,\nRegarding your SaaS Enterprise renewal (Contract #771):\nLast year in 2025 you paid $4,000 for 50 seats. How many seats do you need so we can price your 2026 annual subscription cost?\nRegards, Billing Desk (Ext. 808)",
        "What is the 2026 annual subscription cost?", 1
    ),
    (
        "Ticket #5092 — Damaged Dell UltraSharp (Purchased for 650 EUR on 2026-09-01)\nPlease upload 3 photos of the cracked screen so our team can decide on a refund amount.\nSupport HQ | 2400 Espoo",
        "What is the approved refund amount?", 1
    ),
    (
        "Hi Alex,\nFor the 12x ThinkPad laptops (worth 14,000 EUR), standard ground shipping is 80 EUR. I am still waiting on DHL to quote the express air freight.\nLet me know by 17:00.\n-- Janne, Account Mgr (#ID-302)",
        "How much is express air freight?", 1
    ),
    (
        "Invoice #INV-9001 Draft:\nConsulting hours: 40h at our standard rate. Accounting is still reviewing tax deductions before we calculate the total balance due.\nIBAN FI2112345600000785.",
        "What is the total balance due?", 1
    ),
    (
        "Re: Booth rental for Slush 2026\nPenalty for late cancellation after Oct 15 is 500 EUR. Contact sales@slush.org to get the early-bird booth fee sheet.",
        "What is the early-bird booth fee?", 1
    ),
    (
        "Automated Alert:\nAWS Cloud usage for Sept 2026 (Account 449102938) exceeded your $500.00 budget alert threshold. Final invoice will be billed on Oct 3.",
        "What was the total charged to the card?", 1
    ),
]

# GPU warm-up
_ = extract_mid_layer_vector("Warmup.", "Price?")
if torch.cuda.is_available(): torch.cuda.synchronize()

y_true = [label for _, _, label in messy_holdout_emails]
preds_raw, times_raw, raw_outputs = [], [], []
preds_mute, times_mute, mute_risks = [], [], []

for ctx, q, _ in messy_holdout_emails:
    # 1. Baseline: no Mute, just prompt the model to say NULL when unsure.
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0 = time.perf_counter()
    
    msg_raw = [
        {"role": "system", "content": "Extract the exact requested value from the email. Be brief. If the value is not explicitly stated, output ONLY the word NULL."},
        {"role": "user", "content": f"Email: {ctx}\nQuestion: {q}"}
    ]
    inp_raw = tokenizer.apply_chat_template(
        msg_raw, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        gen_raw = model.generate(**inp_raw, max_new_tokens=15, do_sample=False)
    ans_raw = tokenizer.decode(gen_raw[0, inp_raw["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    
    if torch.cuda.is_available(): torch.cuda.synchronize()
    times_raw.append((time.perf_counter() - t0) * 1000)
    
    abstained = any(k in ans_raw.upper() for k in ["NULL", "NONE", "NOT STATED", "UNKNOWN", "N/A"])
    preds_raw.append(1 if abstained else 0)
    raw_outputs.append(ans_raw)

    # 2. Treatment: Mute attached (reuses the Layer-15 probe trained in Cell 6).
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0 = time.perf_counter()
    
    msg_mute = [
        {"role": "system", "content": "Extract the exact requested value from the email. Be brief."},
        {"role": "user", "content": f"Email: {ctx}\nQuestion: {q}"}
    ]
    inp_mute = tokenizer.apply_chat_template(
        msg_mute, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        out_mute = model(**inp_mute, output_hidden_states=True)
    vec = out_mute.hidden_states[TARGET_LAYER][0, -1, :].to(torch.float32).cpu().numpy().reshape(1, -1)
    risk = probe.predict_proba(vec)[0, 1]
    mute_risks.append(risk)
    
    if risk >= 0.60:
        pred_m = 1  # breaker trips; skip model.generate()
    else:
        pred_m = 0
        with torch.no_grad():
            _ = model.generate(**inp_mute, max_new_tokens=15, do_sample=False)
            
    if torch.cuda.is_available(): torch.cuda.synchronize()
    times_mute.append((time.perf_counter() - t0) * 1000)
    preds_mute.append(pred_m)

# Print the comparison
tn_r, fp_r, fn_r, tp_r = confusion_matrix(y_true, preds_raw).ravel()
tn_m, fp_m, fn_m, tp_m = confusion_matrix(y_true, preds_mute).ravel()

pass_ms = np.mean([t for t, p in zip(times_mute, preds_mute) if p == 0]) if (8 - fp_m + fn_m) > 0 else 0
block_ms = np.mean([t for t, p in zip(times_mute, preds_mute) if p == 1]) if (tp_m + fp_m) > 0 else 0

print("=== ⚡ MUTE live test: short-sentence probe vs 16 unseen multi-line dirty emails ===")
print(f"1. Raw model (prompt says NULL) | accuracy: {accuracy_score(y_true, preds_raw)*100:5.1f}% | traps blocked: {tp_r}/8 | false blocks: {fp_r}/8 | avg: {np.mean(times_raw):5.1f} ms")
print(f"2. Mute (Layer 15 breaker)      | accuracy: {accuracy_score(y_true, preds_mute)*100:5.1f}% | traps blocked: {tp_m}/8 | false blocks: {fp_m}/8 | pass: {pass_ms:5.1f} ms / block: {block_ms:4.1f} ms")

print("\n--- Caught in the act: what did the raw model make up when told to say NULL? ---")
for i in range(8, 16):
    print(f"Trap #{i-7} | raw output: '{raw_outputs[i]:<15}' (blocked: {preds_raw[i]==1}) | Mute risk: {mute_risks[i]:.3f} (blocked: {preds_mute[i]==1})")


In [ ]:
import time
import numpy as np
import torch
from sklearn.metrics import confusion_matrix, accuracy_score

# 1. Internal exception used to physically interrupt the forward pass at Layer 15.
class _MuteStopException(Exception):
    def __init__(self, risk_score: float):
        self.risk_score = risk_score

def generate_with_mute_hook(context: str, question: str, threshold: float = 0.60):
    """
    A single model.generate() call:
    - When the prefill reaches Layer 15, a forward hook computes the probe score in place.
    - If risk >= threshold, raise to interrupt (skips layers 16..24 and all decoding).
    - If risk < threshold, continue seamlessly through layers 16..24 and generate (no re-prefill).
    """
    messages = [
        {"role": "system", "content": "Extract the exact requested value from the email. Be brief."},
        {"role": "user", "content": f"Email: {context}\nQuestion: {question}"}
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
    ).to(model.device)
    
    state = {"risk": None, "prefill_checked": False}
    
    # Fix: Qwen3_5Model's text layers live under model.model.language_model.layers
    decoder_layers = (
        model.model.language_model.layers
        if hasattr(model.model, "language_model")
        else model.model.layers
    )
    # TARGET_LAYER = 15 maps to index 14 (hidden_states[0] is the embedding layer).
    target_module = decoder_layers[TARGET_LAYER - 1]
    
    def layer15_hook(module, args, output):
        # Check once on the first forward pass (prefill), then skip during token-by-token decode.
        if not state["prefill_checked"]:
            state["prefill_checked"] = True
            hidden = output[0] if isinstance(output, tuple) else output
            last_token_vec = (hidden[0, -1, :] if hidden.ndim == 3 else hidden[-1, :])
            vec_np = last_token_vec.to(torch.float32).cpu().numpy().reshape(1, -1)
            
            risk = float(probe.predict_proba(vec_np)[0, 1])
            state["risk"] = risk
            if risk >= threshold:
                raise _MuteStopException(risk)
                
    handle = target_module.register_forward_hook(layer15_hook)
    
    try:
        with torch.no_grad():
            gen_ids = model.generate(**inputs, max_new_tokens=15, do_sample=False)
        ans = tokenizer.decode(gen_ids[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        return {"blocked": False, "answer": ans, "risk": state["risk"]}
    except _MuteStopException as e:
        return {"blocked": True, "answer": None, "risk": e.risk_score}
    finally:
        # Always remove the hook so the model is not polluted.
        handle.remove()

# 2. Abstention detection (independent "NA" word-boundary match to avoid misses).
def is_raw_abstention(text: str) -> bool:
    clean = text.strip().upper().rstrip(".")
    if clean in {"NULL", "NONE", "NA", "N/A", "UNKNOWN", "NOT STATED", "NOT MENTIONED"}:
        return True
    return any(k in clean for k in ["NULL", "NOT STATED", "NOT MENTIONED", "N/A", "NO INFORMATION", "NOT PROVIDED"])

# GPU warm-up
_ = generate_with_mute_hook("Warmup.", "Price?")
if torch.cuda.is_available(): torch.cuda.synchronize()

# 3. Fair per-email evaluation of all 16 emails.
y_true = [label for _, _, label in messy_holdout_emails]
preds_raw, times_raw, ans_raw_list = [], [], []
preds_mute, times_mute, ans_mute_list, risks_mute = [], [], [], []

print("=== 🔬 Per-email check: full outputs and timings for 16 dirty emails (0=normal, 1=missing trap) ===")
print(f"{'ID':<4} | {'true':<4} | {'raw output':<16} | {'raw pred':<8} | {'raw ms':<8} | {'Mute risk':<9} | {'Mute output/action':<16} | {'Mute ms'}")
print("-" * 105)

for idx, (ctx, q, true_label) in enumerate(messy_holdout_emails):
    # --- raw model ---
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0 = time.perf_counter()
    
    msg_raw = [
        {"role": "system", "content": "Extract the exact requested value from the email. Be brief. If the value is not explicitly stated, output ONLY the word NULL."},
        {"role": "user", "content": f"Email: {ctx}\nQuestion: {q}"}
    ]
    inp_raw = tokenizer.apply_chat_template(
        msg_raw, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        gen_raw = model.generate(**inp_raw, max_new_tokens=15, do_sample=False)
    ans_r = tokenizer.decode(gen_raw[0, inp_raw["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    
    if torch.cuda.is_available(): torch.cuda.synchronize()
    ms_r = (time.perf_counter() - t0) * 1000
    
    pred_r = 1 if is_raw_abstention(ans_r) else 0
    preds_raw.append(pred_r)
    times_raw.append(ms_r)
    ans_raw_list.append(ans_r)

    # --- hook-based Mute (no re-prefill + true Layer-15 interrupt) ---
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0 = time.perf_counter()
    
    res_m = generate_with_mute_hook(ctx, q, threshold=0.60)
    
    if torch.cuda.is_available(): torch.cuda.synchronize()
    ms_m = (time.perf_counter() - t0) * 1000
    
    pred_m = 1 if res_m["blocked"] else 0
    preds_mute.append(pred_m)
    times_mute.append(ms_m)
    risks_mute.append(res_m["risk"])
    ans_m_display = "[BLOCKED]" if res_m["blocked"] else str(res_m["answer"])
    ans_mute_list.append(ans_m_display)
    
    raw_status = "block(1)" if pred_r == 1 else "pass(0)"
    print(f"#{idx+1:<3} | {true_label:<4} | {ans_r[:16]:<16} | {raw_status:<8} | {ms_r:5.1f} ms | {res_m['risk']:8.3f} | {ans_m_display[:16]:<16} | {ms_m:5.1f} ms")

# 4. Summary stats (fixed pass_ms comprehension; report mean and median).
tn_r, fp_r, fn_r, tp_r = confusion_matrix(y_true, preds_raw).ravel()
tn_m, fp_m, fn_m, tp_m = confusion_matrix(y_true, preds_mute).ravel()

pass_times_m = [t for t, p in zip(times_mute, preds_mute) if p == 0]
block_times_m = [t for t, p in zip(times_mute, preds_mute) if p == 1]

pass_times_r_on_m_pass = [tr for tr, pm in zip(times_raw, preds_mute) if pm == 0]
pass_times_r_on_m_block = [tr for tr, pm in zip(times_raw, preds_mute) if pm == 1]

print("\n=== 📊 Corrected end-to-end fair benchmark (hook-level single forward + full stats) ===")
print(f"1. Raw baseline | accuracy: {accuracy_score(y_true, preds_raw)*100:5.1f}% | trap recall: {tp_r}/8 ({tp_r/8*100:4.1f}%) | false-block rate: {fp_r}/8 ({fp_r/8*100:4.1f}%)")
print(f"   - all 16 end-to-end: mean {np.mean(times_raw):5.1f} ms | median {np.median(times_raw):5.1f} ms | total {np.sum(times_raw):6.1f} ms")

print(f"\n2. Mute (Hook) | accuracy: {accuracy_score(y_true, preds_mute)*100:5.1f}% | trap recall: {tp_m}/8 ({tp_m/8*100:4.1f}%) | false-block rate: {fp_m}/8 ({fp_m/8*100:4.1f}%)")
print(f"   - all 16 end-to-end: mean {np.mean(times_mute):5.1f} ms | median {np.median(times_mute):5.1f} ms | total {np.sum(times_mute):6.1f} ms")
print(f"   - pass path ({len(pass_times_m)} emails): Mute mean {np.mean(pass_times_m):5.1f} ms (median {np.median(pass_times_m):5.1f} ms) vs raw-on-same {np.mean(pass_times_r_on_m_pass):5.1f} ms")
print(f"   - block path ({len(block_times_m)} emails): Mute mean {np.mean(block_times_m):5.1f} ms (median {np.median(block_times_m):5.1f} ms) vs raw-on-same {np.mean(pass_times_r_on_m_block):5.1f} ms")


In [ ]:
# If `datasets` isn't installed locally, uncomment the next line and run once:
!pip install -q datasets

from datasets import load_dataset
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix

print("Downloading and filtering real human-labeled samples that mention money/numbers...")
ds = load_dataset("rajpurkar/squad_v2", split="train")

real_grounded = []   # Label 0: the paragraph has the real answer
real_ungrounded = [] # Label 1: human-built decoy traps (numbers/money present, but the asked answer is missing)

money_keywords = ("$", "usd", "eur", "price", "cost", "total", "fee", "paid", "amount", "million", "billion")

for row in ds:
    ctx = row["context"]
    q = row["question"]
    # Keep only real passages whose question/context involve money/cost/quantity and contain digits.
    if any(k in q.lower() for k in money_keywords) and any(char.isdigit() for char in ctx):
        is_missing = len(row["answers"]["text"]) == 0
        if not is_missing and len(real_grounded) < 150:
            real_grounded.append((ctx, q, 0))
        elif is_missing and len(real_ungrounded) < 150:
            real_ungrounded.append((ctx, q, 1))
    if len(real_grounded) == 150 and len(real_ungrounded) == 150:
        break

print(f"Got 300 real samples (150 pos / 150 neg)! Avg paragraph length: {np.mean([len(x[0].split()) for x in real_grounded]):.0f} words")

# Strict split: 200 train | 50 threshold-calibration | 50 final blind test
train_split = real_grounded[:100] + real_ungrounded[:100]
val_split   = real_grounded[100:125] + real_ungrounded[100:125]
test_split  = real_grounded[125:150] + real_ungrounded[125:150]

print(f"Extracting Layer {TARGET_LAYER} vectors (300 samples, ~15 s on local GPU)...")
X_tr = np.array([extract_mid_layer_vector(c, q) for c, q, _ in train_split])
y_tr = np.array([y for _, _, y in train_split])

X_val = np.array([extract_mid_layer_vector(c, q) for c, q, _ in val_split])
y_val = np.array([y for _, _, y in val_split])

X_te = np.array([extract_mid_layer_vector(c, q) for c, q, _ in test_split])
y_te = np.array([y for _, _, y in test_split])

# 1. Train the probe on 200 real samples.
real_probe = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=1000))
real_probe.fit(X_tr, y_tr)

# 2. Pick the best threshold on the 50-sample calibration set (Youden's J: maximize TPR - FPR).
val_probs = real_probe.predict_proba(X_val)[:, 1]
best_thresh = 0.50
best_j = -1.0
for th in np.linspace(0.1, 0.9, 81):
    preds_v = (val_probs >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, preds_v).ravel()
    j_score = (tp / (tp + fn)) - (fp / (fp + tn))
    if j_score > best_j:
        best_j = j_score
        best_thresh = th

# 3. Final evaluation on the 50 never-seen blind-test samples.
test_probs = real_probe.predict_proba(X_te)[:, 1]
test_preds = (test_probs >= best_thresh).astype(int)
tn, fp, fn, tp = confusion_matrix(y_te, test_preds).ravel()

print("\n=== 📈 300 real long-passage (avg ~120 words) blind-test results ===")
print(f"Best threshold from calibration: {best_thresh:.2f}")
print(f"Blind-test AUROC (ranking):      {roc_auc_score(y_te, test_probs):.3f}")
print(f"Blind-test accuracy:             {accuracy_score(y_te, test_preds)*100:.1f}%")
print(f"Trap recall:                     {tp}/25 ({tp/25*100:.1f}%) | normal false-block rate (FPR): {fp}/25 ({fp/25*100:.1f}%)")


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

# ---- Improved protocol, step 1: stop trusting the hardcoded Layer 15 on real data.
# Requires the 300-sample cell above (defines train_split / val_split / test_split).
print("Extracting all 25 layers for train/val/test (one forward pass per sample)...")
X_tr_all  = np.array([extract_all_layers_at_once(c, q) for c, q, _ in train_split])  # (200, 25, 1024)
X_val_all = np.array([extract_all_layers_at_once(c, q) for c, q, _ in val_split])    # (50, 25, 1024)
X_te_all  = np.array([extract_all_layers_at_once(c, q) for c, q, _ in test_split])   # (50, 25, 1024)
y_tr  = np.array([y for _, _, y in train_split])
y_val = np.array([y for _, _, y in val_split])
y_te  = np.array([y for _, _, y in test_split])

layer_aucs = []
for l in range(25):
    p = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=1000))
    auc = cross_val_score(p, X_tr_all[:, l, :], y_tr, cv=5, scoring="roc_auc").mean()
    layer_aucs.append(auc)

best_layer = int(np.argmax(layer_aucs))
print(f"\nBest layer by 5-fold AUROC on REAL train split: Layer {best_layer} (AUROC {layer_aucs[best_layer]:.3f})")
print(f"Hardcoded Layer 15 AUROC on real data: {layer_aucs[15]:.3f}")
print("Top-5 layers:", sorted(range(25), key=lambda l: -layer_aucs[l])[:5])
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score, precision_recall_curve, confusion_matrix

L = best_layer

# Train a single probe at the chosen layer, then evaluate ONCE on the untouched test set.
probe_best = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=1000))
probe_best.fit(X_tr_all[:, L, :], y_tr)
val_probs = probe_best.predict_proba(X_val_all[:, L, :])[:, 1]
te_probs  = probe_best.predict_proba(X_te_all[:, L, :])[:, 1]

auc_val = roc_auc_score(y_val, val_probs)
auc_te  = roc_auc_score(y_te, te_probs)
ap_te   = average_precision_score(y_te, te_probs)
print(f"=== Ranking quality @ Layer {L} ===")
print(f"Val  AUROC: {auc_val:.3f}")
print(f"Test AUROC: {auc_te:.3f}  (previous Youden's-J probe @Layer15: 0.872)")
print(f"Test AUPRC: {ap_te:.3f}  (prevalence baseline {y_te.mean():.2f})")

# ---- FPR-capped operating points: the right way to pick a circuit-breaker threshold ----
def pick_threshold_for_fpr_cap(y, probs, cap):
    fpr, tpr, ths = roc_curve(y, probs)
    eligible = fpr <= cap
    if not eligible.any():
        return ths[int(np.argmin(fpr))]
    max_tpr = tpr[eligible].max()
    cand = np.where(eligible & (tpr >= max_tpr - 1e-9))[0]
    return ths[cand[np.argmax(ths[cand])]]

print(f"\n=== FPR-capped operating points (threshold chosen on VAL, scored on TEST) ===")
print(f"{'FPR cap':<8} | {'threshold':<10} | {'test FPR':<9} | {'test recall':<12} | {'test precision':<14} | {'test acc':<9} | {'blocked/50'}")
for cap in (0.05, 0.10, 0.20, 0.35):
    th = pick_threshold_for_fpr_cap(y_val, val_probs, cap)
    pred = (te_probs >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
    fpr_te = fp / (fp + tn)
    rec = tp / (tp + fn)
    prec = tp / (tp + fp) if (tp + fp) else float('nan')
    acc = (tp + tn) / len(y_te)
    print(f"{cap:<8.2f} | {th:<10.3f} | {fpr_te:<9.3f} | {rec:<12.3f} | {prec:<14.3f} | {acc:<9.3f} | {tp+fp}/50")

# ---- bootstrap CI on test AUROC (n=50 -> quantify the uncertainty) ----
rng = np.random.default_rng(0)
boots = []
for _ in range(2000):
    idx = rng.integers(0, len(y_te), len(y_te))
    if len(np.unique(y_te[idx])) < 2:
        continue
    boots.append(roc_auc_score(y_te[idx], te_probs[idx]))
boots = np.array(boots)
print(f"\nTest AUROC bootstrap 95% CI: [{np.percentile(boots, 2.5):.3f}, {np.percentile(boots, 97.5):.3f}]")

# ---- ROC + PR curves ----
fpr, tpr, _ = roc_curve(y_te, te_probs)
prec, rec, _ = precision_recall_curve(y_te, te_probs)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.3))
ax[0].plot(fpr, tpr, lw=2, label=f"AUROC {auc_te:.3f}")
ax[0].plot([0, 1], [0, 1], "k--", lw=0.8)
ax[0].set(xlabel="False Positive Rate", ylabel="True Positive Rate", title=f"ROC @ Layer {L} (real data)")
ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(rec, prec, lw=2, label=f"AUPRC {ap_te:.3f}")
ax[1].axhline(y_te.mean(), color="k", ls="--", lw=0.8, label=f"baseline {y_te.mean():.2f}")
ax[1].set(xlabel="Recall", ylabel="Precision", title=f"Precision-Recall @ Layer {L}")
ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================================
# Reusable Mute evaluation harness
# ----------------------------------------------------------------------------
# Scope discipline (project note): this cell does NOT re-derive generic probe
# findings that the field already established ("hidden states are linearly
# separable", "AUROC is ranking, not a decision metric"). It ONLY enforces the
# steps that are Mute-specific and necessary:
#   1. deterministic + group-aware split  -> no article/thread/contract leakage
#   2. layer selection on TRAIN only      -> the golden layer is data-dependent
#   3. cost-based threshold via FPR caps  -> a circuit breaker is decided by FPR
#   4. honest uncertainty (AUPRC + bootstrap CI) -> Mute's datasets are small
#
# Dependencies (defined in earlier cells): `model`, `tokenizer`,
# `extract_all_layers_at_once` (Cell 16). Samples are [(context, question, label)].
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (roc_auc_score, roc_curve, average_precision_score,
                             precision_recall_curve, confusion_matrix)

_EXTRACT_CACHE = {}   # (context, question) -> (25, 1024) float32; avoids re-extraction


def make_splits(samples, groups=None, seed=0, sizes=(0.6, 0.2, 0.2)):
    """
    Deterministic, group-aware 3-way split. Returns (train_idx, val_idx, test_idx).
    `groups`: optional array of group ids (source article / email thread / contract),
    same length as `samples`. Samples sharing a group never land in different sets.
    """
    n = len(samples)
    g = np.arange(n) if groups is None else np.asarray(groups)

    def _split(groups_arr, train_frac, rs):
        gss = GroupShuffleSplit(n_splits=1, train_size=train_frac, random_state=rs)
        tr, rest = next(gss.split(np.zeros(len(groups_arr)), groups=groups_arr))
        return tr, rest

    tr, rest = _split(g, sizes[0], seed)
    va, te = _split(g[rest], sizes[1] / (sizes[1] + sizes[2]), seed + 1)
    return tr, rest[va], rest[te]


def _extract(samples):
    X = []
    for ctx, q, _ in samples:
        key = (ctx, q)
        if key not in _EXTRACT_CACHE:
            _EXTRACT_CACHE[key] = extract_all_layers_at_once(ctx, q)
        X.append(_EXTRACT_CACHE[key])
    return np.stack(X)   # (N, 25, 1024)


def _pick_threshold_for_fpr_cap(y, probs, cap):
    """Smallest FPR <= cap operating point (maximize recall under the FPR budget)."""
    fpr, tpr, ths = roc_curve(y, probs)
    eligible = fpr <= cap
    if not eligible.any():
        return ths[int(np.argmin(fpr))]
    max_tpr = tpr[eligible].max()
    cand = np.where(eligible & (tpr >= max_tpr - 1e-9))[0]
    return ths[cand[np.argmax(ths[cand])]]


def run_eval(samples, groups=None, layer=None, seed=0, sizes=(0.6, 0.2, 0.2),
             fpr_caps=(0.05, 0.10, 0.20), C=0.1, plot=True):
    """
    End-to-end Mute evaluation over `samples` = [(context, question, label), ...].
    label 0 = answer present (grounded, should PASS); label 1 = answer missing (should BLOCK).
    layer: probe layer 0..24. If None, the best layer is selected on the TRAIN split only.
    Returns a dict with ranking + decision metrics (and prints them).

    Example:
        res = run_eval(samples, groups=source_ids, seed=0)
        res = run_eval(samples, groups=source_ids, layer=20, fpr_caps=(0.05, 0.10))
    """
    tr, va, te = make_splits(samples, groups=groups, seed=seed, sizes=sizes)
    y_tr = np.array([samples[i][2] for i in tr])
    y_va = np.array([samples[i][2] for i in va])
    y_te = np.array([samples[i][2] for i in te])
    print(f"Split: train={len(tr)} val={len(va)} test={len(te)}  (seed={seed}, group-aware={groups is not None})")

    X_tr = _extract([samples[i] for i in tr])
    X_va = _extract([samples[i] for i in va])
    X_te = _extract([samples[i] for i in te])

    # --- layer selection (train-only; the golden layer is data-dependent) ---
    cv_folds = max(2, min(5, int(np.bincount(y_tr).min())))
    layer_aucs = []
    for l in range(25):
        p = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=1000))
        layer_aucs.append(cross_val_score(p, X_tr[:, l, :], y_tr, cv=cv_folds, scoring="roc_auc").mean())
    best = int(np.argmax(layer_aucs))
    if layer is None:
        layer = best
    print(f"Layer: using {layer}  (train CV-AUROC best={layer_aucs[best]:.3f} @L{best}, L15={layer_aucs[15]:.3f})")

    # --- train one probe at the chosen layer, evaluate ONCE on the untouched test set ---
    probe = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=1000))
    probe.fit(X_tr[:, layer, :], y_tr)
    va_probs = probe.predict_proba(X_va[:, layer, :])[:, 1]
    te_probs = probe.predict_proba(X_te[:, layer, :])[:, 1]

    auc_va = roc_auc_score(y_va, va_probs)
    auc_te = roc_auc_score(y_te, te_probs)
    auprc_te = average_precision_score(y_te, te_probs)

    # --- cost-based operating points (threshold picked on VAL, scored on TEST) ---
    rows = []
    for cap in fpr_caps:
        th = _pick_threshold_for_fpr_cap(y_va, va_probs, cap)
        pred = (te_probs >= th).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
        fpr = fp / (fp + tn)
        rec = tp / (tp + fn)
        prec = tp / (tp + fp) if (tp + fp) else float("nan")
        rows.append((cap, th, fpr, rec, prec, (tp + tn) / len(y_te), tp + fp))

    # --- bootstrap CI on test AUROC (small n -> quantify the uncertainty) ---
    rng = np.random.default_rng(seed + 2)
    boots = []
    for _ in range(2000):
        b = rng.integers(0, len(y_te), len(y_te))
        if len(np.unique(y_te[b])) < 2:
            continue
        boots.append(roc_auc_score(y_te[b], te_probs[b]))
    boots = np.asarray(boots)
    ci = (np.percentile(boots, 2.5), np.percentile(boots, 97.5))

    print(f"Ranking: val AUROC {auc_va:.3f} | test AUROC {auc_te:.3f} "
          f"(95% CI {ci[0]:.3f}-{ci[1]:.3f}) | test AUPRC {auprc_te:.3f} (baseline {y_te.mean():.2f})")
    print("Operating points (threshold from VAL, scored on TEST):")
    print(f"  {'FPR cap':<8}{'thr':<8}{'test FPR':<10}{'recall':<9}{'precision':<11}{'acc':<8}{'blocked'}")
    for cap, th, fpr, rec, prec, acc, blocked in rows:
        print(f"  {cap:<8.2f}{th:<8.3f}{fpr:<10.3f}{rec:<9.3f}{prec:<11.3f}{acc:<8.3f}{blocked}")

    if plot:
        fpr, tpr, _ = roc_curve(y_te, te_probs)
        prec_curve, rec_curve, _ = precision_recall_curve(y_te, te_probs)
        fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
        ax[0].plot(fpr, tpr, lw=2, label=f"AUROC {auc_te:.3f}")
        ax[0].plot([0, 1], [0, 1], "k--", lw=0.8)
        ax[0].set(xlabel="False Positive Rate", ylabel="True Positive Rate", title=f"ROC @ layer {layer}")
        ax[0].legend(); ax[0].grid(alpha=0.3)
        ax[1].plot(rec_curve, prec_curve, lw=2, label=f"AUPRC {auprc_te:.3f}")
        ax[1].axhline(y_te.mean(), color="k", ls="--", lw=0.8, label=f"baseline {y_te.mean():.2f}")
        ax[1].set(xlabel="Recall", ylabel="Precision", title=f"Precision-Recall @ layer {layer}")
        ax[1].legend(); ax[1].grid(alpha=0.3)
        plt.tight_layout(); plt.show()

    return {
        "layer": layer, "best_layer": best, "layer_aucs": layer_aucs,
        "auc_val": auc_va, "auc_test": auc_te, "auprc_test": auprc_te,
        "auc_test_ci95": ci,
        "operating_points": [dict(fpr_cap=c, threshold=t, test_fpr=f, recall=r, precision=p, accuracy=a)
                             for c, t, f, r, p, a, _ in rows],
    }


In [ ]:
# ============================================================================
# Contract example: what run_eval actually consumes (the "contract" in concrete form)
# ----------------------------------------------------------------------------
# samples = [(email body context, question, label), ...]
#   label 0 = the email has the answer             -> the probe should PASS
#   label 1 = the email lacks the answer (decoys)  -> the probe should BLOCK
# Feed in real emails + questions + 0/1 labels; splitting / layer selection / train /
# thresholding / reporting are all automatic.
# ============================================================================

demo_samples = [
    # --- has answer (label=0) ---
    ("Hi, the catering deposit is 350 EUR, due Thursday.", "What is the deposit?", 0),
    ("Customs clearance fee came to $125.00. Storage waived.", "What was the customs fee?", 0),
    ("Invoice #INV-9001: total balance due is 4,800 EUR.", "What is the total due?", 0),
    ("The early-bird booth fee is 1,950 EUR (includes 4 passes).", "What is the booth fee?", 0),
    # --- no answer, decoy only (label=1) ---
    ("Caterer is still calculating the deposit. (45 guests attending)", "What is the deposit?", 1),
    ("Customs is still holding the shipment, no fee told yet. (weight 620 kg)", "What was the customs fee?", 1),
    ("Accounting is reviewing deductions before totalling. (40 consulting hours)", "What is the total due?", 1),
    ("Contact sales for the fee sheet. (late-cancellation penalty 500 EUR)", "What is the booth fee?", 1),
]

# groups (optional): tag each sample with a "source id" so one source never spans train/test.
# For real data use: email thread id (aeslc) or contract id (cuad). Here we demo the no-groups split:
tr, va, te = make_splits(demo_samples, seed=0)
print("No-groups split -> train", len(tr), "val", len(va), "test", len(te), "samples")

# With groups (uncomment once you have real source ids):
# demo_groups = ["thread-1"] * 4 + ["thread-2"] * 4
# tr, va, te = make_splits(demo_samples, groups=demo_groups, seed=0)

# ⚠️ These 8 are too few for run_eval to be meaningful, so the real call is commented out.
# Once you pull a few hundred real samples from aeslc, switch to:
#   res = run_eval(my_samples, groups=my_source_ids, seed=0)
# It will: split -> pick the best layer on TRAIN only -> train the probe ->
# threshold by "allowed FPR" -> blind test + tables + plots.


In [ ]:
# ============================================================================
# Build samples from real Enron emails (aeslc) automatically — no hand-writing 200 of them
# ----------------------------------------------------------------------------
# Labeling rule (automatic, no humans). Both sides check "is there a value near the field word":
#   grounded   (label 0): field word appears AND a money amount sits within 30 chars
#   ungrounded (label 1): field word appears but NO money within 30 chars (value not stated),
#                         while the email has decoy money elsewhere
# Two details that matter:
#   1) whole-word boundary \bfield\b — otherwise "price" matches "prices"
#   2) change/discount blacklist — rejects "price down by $5" (a change, not a price)
# group id = email index i (pos/neg samples from the same email stay in the same split).
# ============================================================================
import re
import numpy as np
from datasets import load_dataset

# ---- 1. Load real Enron emails ----
ds = load_dataset("Yale-LILY/aeslc", split="train")
print("columns:", ds.column_names, "| n emails:", len(ds))

body_col = "email_body" if "email_body" in ds.column_names else ds.column_names[0]

# ---- 2. Money regex + field -> question templates ----
MONEY_RE = re.compile(r"\$\s?\d[\d,.]*(?:\s?[kKmM])?|\d[\d,.]*\s?(?:EUR|USD|€|dollars|euros)")
CHANGE_RE = re.compile(
    r"\b(down|up|by|off|save|savings|discount|increase|decrease|lower|higher|"
    r"reduce|reduction|less|more)\b",
    re.IGNORECASE,
)

FIELDS = [
    ("deposit", "What is the deposit?"),
    ("total", "What is the total amount?"),
    ("fee", "What is the fee?"),
    ("price", "What is the price?"),
    ("cost", "What is the cost?"),
    ("amount due", "What is the amount due?"),
    ("balance", "What is the balance?"),
    ("refund", "What is the refund amount?"),
    ("payment", "What is the payment amount?"),
]

def _field_pattern(fword):
    # "amount due" -> \bamount\b\s+\bdue\b ; whole-word boundary
    return r"\b" + r"\b\s+\b".join(re.escape(p) for p in fword.split()) + r"\b"

def has_value(body, fword, radius=30):
    """Is there a money amount within `radius` chars of the field word (and not a change/discount)?"""
    m = re.search(_field_pattern(fword), body, re.IGNORECASE)
    if not m:
        return False
    # look forward
    after = body[m.end(): m.end() + radius]
    mm = MONEY_RE.search(after)
    if mm and not CHANGE_RE.search(after[:mm.start()]):
        return True
    # look backward ("$350 deposit")
    before = body[max(0, m.start() - radius): m.start()]
    mb = MONEY_RE.search(before)
    if mb and not CHANGE_RE.search(before[mb.end():]):
        return True
    return False

def mentions_without_value(body, fword):
    """Field word present but no value next to it (value not stated); email has decoy money elsewhere."""
    if not re.search(_field_pattern(fword), body, re.IGNORECASE):
        return False
    if has_value(body, fword):
        return False
    return bool(MONEY_RE.search(body))

# ---- 3. Scan real emails, build pos/neg samples ----
MAX_LEN, MIN_LEN, MAX_EACH = 600, 40, 150
grounded, ungrounded = [], []

for i, row in enumerate(ds):
    body = str(row[body_col]).strip()
    if len(body) < MIN_LEN:
        continue
    body = body[:MAX_LEN]
    for fword, q in FIELDS:
        if len(grounded) < MAX_EACH and has_value(body, fword):
            grounded.append((body, q, 0, i))
        elif len(ungrounded) < MAX_EACH and mentions_without_value(body, fword):
            ungrounded.append((body, q, 1, i))
    if len(grounded) >= MAX_EACH and len(ungrounded) >= MAX_EACH:
        break

# Balance both classes + fixed-seed shuffle
n = min(len(grounded), len(ungrounded))
combined = grounded[:n] + ungrounded[:n]
order = np.random.default_rng(0).permutation(len(combined))
combined = [combined[k] for k in order]

my_samples = [(c, q, lab) for c, q, lab, _ in combined]
my_source_ids = [sid for _, _, _, sid in combined]

print(f"\nBuilt: grounded={len(grounded)} ungrounded={len(ungrounded)} -> {len(my_samples)} samples (shuffle seed=0)")
print("Label distribution:", np.bincount([lab for _, _, lab in my_samples]).tolist())

print("\n--- 3 grounded examples (field word + adjacent amount) ---")
shown = 0
for c, q, lab in my_samples:
    if lab == 0 and shown < 3:
        print(f"  Q: {q}\n  email: {c[:150]}\n")
        shown += 1

print("--- 3 ungrounded examples (field word present, no adjacent amount) ---")
shown = 0
for c, q, lab in my_samples:
    if lab == 1 and shown < 3:
        print(f"  Q: {q}\n  email: {c[:150]}\n")
        shown += 1

# ---- 4. run_eval lives in the next cell ----


In [ ]:
# Diagnostic: print full sample text + field-word context + money matches, to eyeball the labels.
def show(label, k):
    shown = 0
    for c, q, lab in my_samples:
        if lab != label or shown >= k:
            continue
        fword = next((f for f, qq in FIELDS if qq == q), "?")
        tag = "GROUNDED(has answer)" if label == 0 else "UNGROUNDED(no answer)"
        print(f"\n===== {tag} | Q: {q} | field word '{fword}' =====")
        print("full text (<=600 chars):")
        print(c)
        m = re.search(_field_pattern(fword), c, re.IGNORECASE)
        if m:
            print(f">>> field word appears at: ...{c[max(0, m.start()-60):m.end()+60]}...")
        print(">>> all money matches:", MONEY_RE.findall(c))
        shown += 1

show(0, 3)   # grounded: field word + value both present
show(1, 4)   # ungrounded: key check — field word present, but no value next to it


In [ ]:
# Full evaluation on the real Enron email samples (group-aware split + train-only layer selection + FPR threshold + blind test).
# NOTE: aeslc regex auto-labeling has a precision ceiling — treat these numbers as a rough baseline, not gold.
res = run_eval(my_samples, groups=my_source_ids, seed=0)


In [ ]:
# ============================================================================
# cuad-qa: real commercial contracts + human gold labels (answerable vs unanswerable)
# ----------------------------------------------------------------------------
# Data source = raw CUAD train_separate_questions.json. Newer `datasets` no longer
# supports legacy script datasets, so we read the JSON straight from GitHub's zip.
# "Unanswerable" = that question's answers list is empty.
# Contract paragraphs are long (median ~45k chars), so we take windows, not whole paragraphs:
#   answerable   -> 1500-char window centered on the answer position (answer stays inside)
#   unanswerable -> random-position 1500-char window (no answer anywhere in the paragraph)
# group id = contract title, so questions from the same contract never split across sets.
# ============================================================================
import urllib.request, zipfile, io, json
import numpy as np

URL = "https://github.com/TheAtticusProject/cuad/raw/main/data.zip"
WINDOW, MAX_EACH, SEED = 1500, 150, 0

z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(URL).read()))
j = json.loads(z.read("train_separate_questions.json"))

pos, neg = [], []   # pos=answerable, neg=unanswerable
for ex in j["data"]:
    title = ex.get("title", "").strip()
    for p in ex["paragraphs"]:
        ctx = p["context"]
        for qa in p["qas"]:
            q = qa["question"].strip()
            if qa["answers"] and qa["answers"][0]["answer_start"] < len(ctx):
                pos.append((ctx, q, title, qa["answers"][0]["answer_start"]))
            elif not qa["answers"]:
                neg.append((ctx, q, title))
print(f"Raw: answerable={len(pos)} unanswerable={len(neg)}")

rng = np.random.default_rng(SEED)
rng.shuffle(pos)
rng.shuffle(neg)

samples, source_ids = [], []
for ctx, q, title, s in pos[:MAX_EACH]:
    lo = max(0, s - WINDOW // 2)
    samples.append((ctx[lo:lo + WINDOW], q, 0))
    source_ids.append(title)
for ctx, q, title in neg[:MAX_EACH]:
    off = int(rng.integers(0, max(1, len(ctx) - WINDOW)))
    samples.append((ctx[off:off + WINDOW], q, 1))
    source_ids.append(title)

order = rng.permutation(len(samples))
samples = [samples[k] for k in order]
source_ids = [source_ids[k] for k in order]

print(f"\ncuad-qa samples: {len(samples)} total, label distribution {np.bincount([lab for _, _, lab in samples]).tolist()}")
for c, q, lab in samples[:2]:
    print(f"\n[{'answerable' if lab == 0 else 'unanswerable'}] Q: {q}\n  context: {c[:220]}...")

# Run the full evaluation on gold labels.
res = run_eval(samples, groups=source_ids, seed=0)
